In [ ]:
!pip install kagglehub
import kagglehub

path = kagglehub.dataset_download("andrewmvd/road-sign-detection")

print("Path to dataset files:", path)

import os
os.environ["HF_TOKEN"] = "***************your_huggingface_token***************"
import io
import shutil
import requests
!pip install matplotlib
import matplotlib.pyplot as plt
import numpy as np
from glob import glob
from PIL import Image
from tqdm import tqdm
 
!pip install lxml
from lxml import etree

!pip install ultralytics
from ultralytics import YOLO

!pip install transformers pillow torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch

from PIL import Image, ImageDraw, ImageFont

!pip install pyttsx3
import pyttsx3
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Nazwa karty: {torch.cuda.get_device_name(0)}")

!ls $path

import cv2
!pip install gtts
from gtts import gTTS
from IPython.display import Audio, display, clear_output
!pip install ipywidgets
import ipywidgets as widgets

Path to dataset files: /Users/sebastianstarzec/.cache/kagglehub/datasets/andrewmvd/road-sign-detection/versions/1


--- Logging error ---


UnboundLocalError: cannot access local variable 'child' where it is not associated with a value

Traceback (most recent call last):
  File "/Users/sebastianstarzec/Documents/Magisterka/Semestr 4/projektNlp/.venv/lib/python3.13/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/sebastianstarzec/Documents/Magisterka/Semestr 4/projektNlp/.venv/lib/python3.13/site-packages/IPython/utils/_process_posix.py", line 125, in system
    child = pexpect.spawn(self.sh, args=['-c', cmd])  # Vanilla Pexpect
  File "/Users/sebastianstarzec/Documents/Magisterka/Semestr 4/projektNlp/.venv/lib/python3.13/site-packages/pexpect/pty_spawn.py", line 205, in __init__
    self._spawn(command, args, preexec_fn, dimensions)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sebastianstarzec/Documents/Magisterka/Semestr 4/projektNlp/.venv/lib/python3.13/site-packages/pexpect/pty_spawn.py", line 303, in _spawn
    self.ptyproc = self._spawnpty(self.args, env=self.env,
                   ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
            

  File "/Users/sebastianstarzec/Documents/Magisterka/Semestr 4/projektNlp/.venv/lib/python3.13/site-packages/IPython/utils/_process_posix.py", line 141, in system
    child.sendline(chr(3))
    ^^^^^
UnboundLocalError: cannot access local variable 'child' where it is not associated with a value

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.13_1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/logging/__init__.py", line 1155, in emit
    self.flush()
    ~~~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.13/3.13.13_1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/logging/__init__.py", line 1137, in flush
    self.stream.flush()
    ~~~~~~~~~~~~~~~~~^^
OSError: [Errno 9] Bad file descriptor
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/sebastianstarzec/Documents/Magisterka/Semestr 4

In [ ]:
print("All libraries imported successfully!")

In [ ]:
def split_train_test(path: str, train_size: float = 0.8, val_size: float = 0.1, seed: int = None):
    if seed is not None:
        np.random.seed(seed)

    annots = np.array(os.listdir(os.path.join(path, "annotations")))
    imgs = np.array(os.listdir(os.path.join(path, "images")))

    annots.sort()
    imgs.sort()

    train_len = int(len(annots) * train_size)
    val_len = int(len(annots) * val_size)
    test_len = len(annots) - train_len - val_len

    indices = np.arange(len(annots), dtype=int)
    np.random.shuffle(indices)

    train_idx = indices[:train_len]
    val_idx = indices[:train_len + val_len]
    test_idx = indices[train_len+val_len:]

    train_annots = annots[train_idx]
    train_imgs = imgs[train_idx]

    val_annots = annots[val_idx]
    val_imgs = imgs[val_idx]

    test_annots = annots[test_idx]
    test_imgs = imgs[test_idx]

    return (train_annots, train_imgs), (val_annots, val_imgs), (test_annots, test_imgs)

In [ ]:
train, val, test = split_train_test(path, 0.8, 0.1, seed=42)

In [ ]:
def extract_data(xml_file: str):
    tree = etree.parse(xml_file)
    root = tree.getroot()

    img_path = root.find("filename").text

    size = root.find("size")
    width = int(size.find("width").text)
    height = int(size.find("height").text)

    boxes = []
    labels = []

    for obj in root.findall("object"):
        class_name = obj.find("name").text
        bndbox = obj.find("bndbox")

        xmin = int(bndbox.find("xmin").text)
        ymin = int(bndbox.find("ymin").text)
        xmax = int(bndbox.find("xmax").text)
        ymax = int(bndbox.find("ymax").text)

        labels.append(class_name)
        boxes.append([xmin, ymin, xmax, ymax])

    return img_path, width, height, boxes, labels

In [ ]:

def rcnn_to_yolo(img_w, img_h, xmin, ymin, xmax, ymax):
    x_abs = (xmin + xmax) / 2
    y_abs = (ymin + ymax) / 2
    w_abs = xmax - xmin
    h_abs = ymax - ymin

    xc = x_abs / img_w
    yc = y_abs / img_h
    w = w_abs / img_w
    h = h_abs / img_h

    return xc, yc, w, h

In [ ]:
def create_yolo_files(data: tuple, folder: str):
  os.makedirs(f"{folder}/labels", exist_ok=True)
  os.makedirs(f"{folder}/images", exist_ok=True)

  for (annots, imgs) in data:
    for i in tqdm(range(len(annots))):
      img_path, width, height, boxes, labels = extract_data(os.path.join(path, "annotations", annots[i]))

      result = []
      for j in range(len(boxes)):
         xmin, ymin, xmax, ymax = boxes[j]
         class_id = class_dict[labels[j]]

         box = rcnn_to_yolo(width, height, xmin, ymin, xmax, ymax)
         result.append(f"{class_id} {' '.join(str(x) for x in box)}")

      with open(f"{folder}/labels/{i}.txt", "w") as f:
         content = "\n".join(result)
         f.write(content)

      shutil.copy(os.path.join(path, "images", img_path), f"{folder}/images/{i}.jpg")

In [ ]:
class_dict = {"stop": 0, "speedlimit": 1, "crosswalk": 2, "trafficlight": 3}

create_yolo_files([val], "data/val")
create_yolo_files([train], "data/train")
create_yolo_files([test], "data/test")

In [ ]:
def create_yaml(root: str, train: str, valid: str, classes: list):
  """ Create dataset.yaml """
  nc = len(classes)
  content = f"""
path: {root}
train: {train}
val: {valid}

nc: {nc}
names: {classes}
  """
  with open("dataset.yaml", "w") as f:
      f.write(content)
  print("Created dataset.yaml!")

In [ ]:
create_yaml("data", "train", "val", list(class_dict.keys()))

In [ ]:
!cat dataset.yaml

In [ ]:
model = YOLO("yolo26n.pt")

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
model_ocr = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed').to(device)

class_dict = {"stop": 0, "speedlimit": 1, "crosswalk": 2, "trafficlight": 3}

#model.train(data="dataset.yaml", batch=32, epochs=1, imgsz=640, device='mps',workers=0,plots=True)

In [ ]:
best_model = YOLO("runs/detect/train/weights/best.pt")

In [ ]:
test_imgs = glob("data/test/images/*.jpg")

results = best_model.predict(test_imgs[:5], imgsz=320)

fig, axes = plt.subplots(1, len(results), figsize=(15, 5))

for ax, result in zip(axes, results):
    img = result.plot()  
    ax.imshow(img[:, :, ::-1]) 
    ax.axis("off")

plt.suptitle("Model's prediction")
plt.tight_layout()
plt.show()

In [ ]:
def predict_and_show(img_path, yolo_model):
    results = yolo_model.predict(img_path, conf=0.5)
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            class_id = int(box.cls)
            label = yolo_model.names[class_id]

            if label == 'speedlimit':
                crop = img.crop((max(0, x1-5), max(0, y1-5), x2+5, y2+5))
                
                pixel_values = processor(images=crop, return_tensors="pt").pixel_values.to(device)
                generated_ids = model_ocr.generate(pixel_values)
                text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
                
                clean_text = ''.join(filter(str.isdigit, text))
                display_label = f"Limit: {clean_text}" if clean_text else "Sign"
            else:
                display_label = label

            draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
            draw.text((x1, y1 - 10), display_label, fill="red")
    
    return img

In [ ]:
model_path = "runs/detect/train5/weights/best.pt"
trained_model = YOLO(model_path)

test_img = "data/test/images/25.jpg" 

result_img = predict_and_show(test_img, trained_model)

display(result_img)